# 07 Labelling with the construct framework (Claude Sonnet 5, Batch API)

`ANTHROPIC_API_KEY` will return an error because the `.env` file was not uploaded – own Anthropic key can be generated the Anthropic Console -> https://platform.claude.com/settings/keys

## Settings
`SUBREDDIT` = name used in the prompt. `PILOT_N` = number for a pilot run on a random sample (saved in a separate `pilot_<N>` folder).

In [ ]:
import json, os, random, re, time, unicodedata
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import anthropic

# input files, subreddit name in the prompt and output folder
INPUT_FILE     = Path("path/to/Subreddit_testimony_screened.csv")
FRAMEWORK_FILE = Path("path/to/Subreddit_FrameworkForCode.csv")
SUBREDDIT      = "Subreddit"   # Feminism or MensRights
OUT_DIR        = Path("path/to/output_folder")
TEXT_COL = "datapoint"
# exit is in the framework files but not part of the final framework (21 constructs), so it is not coded
DROP_CONSTRUCTS = ["EXIT"]

# model settings
MODEL      = "claude-sonnet-5"
EFFORT     = "high"
MAX_TOKENS = 1000      # max per answer
MAX_CHARS  = 6000      # cut texts to this length
POLL_SECONDS = 60
PILOT_N    = None      # e.g. 50 for a pilot
COST_RATES = {"in": 1.00, "out": 5.00}   # batch price per 1M tokens (for the cost estimate)
EST_OUTPUT_TOKENS = 60                   # rough output per request (for the estimate)

## API key

In [ ]:
# will return an error if the API key is not set in the environment or in ../.env
try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"))
except ImportError:
    pass
# stop if there is no key
if not os.environ.get("ANTHROPIC_API_KEY", "").strip():
    raise RuntimeError("Set the ANTHROPIC_API_KEY environment variable (or put it in ../.env).")
client = anthropic.Anthropic()
print("API key found.")

## Load data and framework
Keeps the rows marked as testimony in step 06 and drops empty texts. In the framework, `Code` = construct name,
`Definition` = text sent to the model. The constructs in `DROP_CONSTRUCTS` (EXIT) are left out, so 21 constructs are coded.

In [ ]:
# load screened data (sep=None so it works with , or ;)
df = pd.read_csv(INPUT_FILE, sep=None, engine="python")
# keep testimonies only, drop empty texts
if "is_testimony" in df.columns:
    df = df[df["is_testimony"].astype(str).str.lower() == "true"]
text = df[TEXT_COL].fillna("").astype(str)
n_empty = int((text.str.strip() == "").sum())
df = df[text.str.strip() != ""]
# random sample for a pilot
if PILOT_N:
    df = df.sample(n=min(PILOT_N, len(df)), random_state=0)
    OUT_DIR = OUT_DIR / f"pilot_{PILOT_N}"
# cut long texts and note which ones were cut
df = df.reset_index(drop=True)
df["text_sent"] = df[TEXT_COL].astype(str).str[:MAX_CHARS]
df["was_truncated"] = df[TEXT_COL].astype(str).str.len() > MAX_CHARS

# load the framework
fw = pd.read_csv(FRAMEWORK_FILE, sep=";").dropna(subset=["Code"])
fw = fw[~fw["Code"].str.strip().isin(DROP_CONSTRUCTS)]   # leave out exit
assert fw["Code"].is_unique and fw["Definition"].notna().all(), "check the framework file"
concepts = [{"label": l.strip(), "definition": d} for l, d in zip(fw["Code"], fw["Definition"])]

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(df):,} datapoints x {len(concepts)} constructs | {n_empty} empty dropped | "
      f"{int(df['was_truncated'].sum())} truncated")

## Prompts, checks and batch functions

In [ ]:
VALID_SCORES = {0.0, 0.5, 1.0}

# system prompt with the scoring rules
SYSTEM = (
    f"You are a qualitative research assistant. Your task is to code Reddit posts from the "
    f"r/{SUBREDDIT} subreddit based on the presence or absence of a particular construct, which is "
    f"part of a set of {len(concepts)} constructs. Score the following Reddit posts or comments using "
    "these rules:\n  0   = the construct is absent\n"
    "  0.5 = the construct is partially present or ambiguous\n  1   = the construct is clearly present\n\n"
    "Additional rules:\n- Base your score ONLY on explicit evidence in the post text.\n"
    "- If score > 0, provide an exact verbatim quote from the post (the shortest span that justifies "
    "the score). Copy it character-for-character.\n- If score = 0, set quote to null.\n"
    "- The score must be exactly 0, 0.5, or 1. No other values.\n"
    "- Respond with valid JSON only - no prose, no markdown fences.")

# user message = construct definition + post
def user_message(definition, text, note=""):
    msg = (f"Score the post below on the following concept:\n{definition}\n\n"
           f"Post:\n{text}\n\nReturn JSON with keys: score, quote.")
    return msg + ("\n\n" + note if note else "")

# extra note added on the retry when the quote was not found
RETRY_NOTE = ("IMPORTANT: In a previous attempt your quote could not be found verbatim in the post. "
              "Copy the quote EXACTLY, character-for-character, from the post text above. If no exact "
              "quote supports a score above 0, return score 0 with quote null.")

def _norm(s):
    # lowercase, same quote marks, no punctuation, single spaces
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s']", " ", s)).strip()

def verify_quote(quote, text):
    # quote has to be in the text; if it has "..." every part has to be in the text
    if quote is None or str(quote).strip().lower() in {"", "null", "none", "nan"}:
        return False
    parts = [p for p in re.split(r"\s*(?:\.\.\.|…)\s*", str(quote)) if _norm(p)]
    return bool(parts) and all(_norm(p) in _norm(text) for p in parts)

def parse_answer(raw, text):
    # read the json answer -> score, quote, quote_verified, flag (flag None = all fine)
    try:
        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", str(raw).strip())
        m = re.search(r"\{.*\}", cleaned, re.DOTALL)
        parsed = json.loads(m.group() if m else cleaned)
    except Exception:
        return dict(score=None, quote=None, quote_verified=None, flag="parse_failed")
    if isinstance(parsed, dict) and "score" not in parsed and len(parsed) == 1:
        parsed = next(iter(parsed.values()))      # in case the answer is nested like {label: {...}}
    try:
        score = float(parsed.get("score"))
    except Exception:
        score = None
    if score not in VALID_SCORES:
        return dict(score=None, quote=None, quote_verified=None, flag="invalid_score")
    quote = parsed.get("quote")
    ok = verify_quote(quote, text) if score > 0 else None
    return dict(score=score, quote=quote, quote_verified=ok, flag="quote_unverified" if ok is False else None)

def make_request(d, c, note=""):
    # one batch request for datapoint d and construct c
    return {"custom_id": f"d{d}__c{c}",
            "params": {"model": MODEL, "max_tokens": MAX_TOKENS, "system": SYSTEM,
                       "output_config": {"effort": EFFORT},
                       "messages": [{"role": "user", "content": user_message(
                           concepts[c]["definition"], df.at[d, "text_sent"], note)}]}}

def submit_or_resume(requests, state_path):
    # reuse the saved batch id if there is one, otherwise submit and save the id
    if state_path.exists():
        batch_id = json.loads(state_path.read_text())["batch_id"]
        print(f"existing batch {batch_id} (delete {state_path.name} to resubmit)")
        return batch_id
    batch_id = client.messages.batches.create(requests=requests).id
    state_path.write_text(json.dumps({"batch_id": batch_id, "n_requests": len(requests),
                                      "submitted_at": datetime.now(timezone.utc).isoformat()}, indent=2))
    print(f"submitted batch {batch_id} ({len(requests):,} requests)")
    return batch_id

def wait_for(batch_id):
    # check the batch status until it has ended
    while True:
        b = client.messages.batches.retrieve(batch_id)
        c = b.request_counts
        print(f"{datetime.now():%H:%M} {b.processing_status}: succeeded {c.succeeded:,}, "
              f"errored {c.errored:,}, processing {c.processing:,}")
        if b.processing_status == "ended":
            return
        time.sleep(POLL_SECONDS)

def collect(batch_id, results, only_if_clean=False):
    # read the batch results into results[(d, c)], returns input / output tokens
    tin = tout = 0
    for r in client.messages.batches.results(batch_id):
        d, c = (int(x[1:]) for x in r.custom_id.split("__"))
        if r.result.type != "succeeded":
            entry = dict(score=None, quote=None, quote_verified=None, flag=f"api_error:{r.result.type}")
        else:
            msg = r.result.message
            tin, tout = tin + msg.usage.input_tokens, tout + msg.usage.output_tokens
            text_block = next((b.text for b in msg.content if b.type == "text"), None)
            entry = parse_answer(text_block, df.at[d, "text_sent"])
        if not only_if_clean or entry["flag"] is None:
            results[(d, c)] = entry
    return tin, tout

## Cost estimate
Counts the input tokens of 40 random requests and uses `EST_OUTPUT_TOKENS` for the output.

In [ ]:
# all datapoint x construct pairs, estimate from 40 random ones
pairs = [(d, c) for d in range(len(df)) for c in range(len(concepts))]
sample = random.Random(0).sample(pairs, min(40, len(pairs)))
avg_in = np.mean([client.messages.count_tokens(model=MODEL, system=SYSTEM,
                  messages=make_request(d, c)["params"]["messages"]).input_tokens for d, c in sample])
est = len(pairs) * (avg_in * COST_RATES["in"] + EST_OUTPUT_TOKENS * COST_RATES["out"]) / 1e6
print(f"{len(pairs):,} requests, estimated cost ~${est:.2f}")

## Submit

In [ ]:
# build all requests and submit (or reuse the saved batch)
batch_id = submit_or_resume([make_request(d, c) for d, c in pairs], OUT_DIR / "_batch_state.json")

## Collect, retry, save
Saves `coded_final.csv` (per construct: `__score`, `__quote`, `__quote_verified`, `__flag`).

In [ ]:
# wait for the batch and read the answers
wait_for(batch_id)
results = {}
tin, tout = collect(batch_id, results)
first_problems = sum(e["flag"] is not None for e in results.values())

# send the problem pairs one more time (with the note if the quote was wrong)
retry = [k for k, e in results.items() if e["flag"] is not None]
if retry:
    rid = submit_or_resume([make_request(d, c, RETRY_NOTE if results[(d, c)]["flag"] == "quote_unverified" else "")
                            for d, c in retry], OUT_DIR / "_retry_batch_state.json")
    wait_for(rid)
    a, b = collect(rid, results, only_if_clean=True)
    tin, tout = tin + a, tout + b

# one set of columns per construct, then save
coded = df.copy()
for c, concept in enumerate(concepts):
    for field in ("score", "quote", "quote_verified", "flag"):
        coded[f"{concept['label']}__{field}"] = [results.get((d, c), {"flag": "missing"}).get(field)
                                                 for d in range(len(df))]
coded.to_csv(OUT_DIR / "coded_final.csv", index=False)

## Summary

In [ ]:
# overall counts
entries = [results.get((d, c), {"score": None, "flag": "missing"})
           for d in range(len(df)) for c in range(len(concepts))]
counts = pd.Series({
    "datapoints": len(df),
    "empty texts dropped": n_empty,
    "truncated datapoints": int(df["was_truncated"].sum()),
    "datapoint-construct pairs": len(entries),
    "problems after first pass (retried)": first_problems,
    "missing after retry": sum(e["score"] is None for e in entries),
    "positive codes (score > 0)": sum((e["score"] or 0) > 0 for e in entries),
    "positive codes with unverified quote": sum(e["flag"] == "quote_unverified" for e in entries),
    "input tokens": tin,
    "output tokens": tout,
    "cost ($)": round((tin * COST_RATES["in"] + tout * COST_RATES["out"]) / 1e6, 2),
}, dtype=object)
display(counts)
counts.to_csv(OUT_DIR / "coding_counts.csv", header=["value"])

# score distribution per construct
dist = pd.DataFrame({k["label"]: pd.to_numeric(coded[f"{k['label']}__score"], errors="coerce")
                     .value_counts(dropna=False).reindex([0, 0.5, 1, np.nan], fill_value=0).values
                     for k in concepts}, index=["0", "0.5", "1", "missing"]).T
display(dist)
dist.to_csv(OUT_DIR / "score_distribution.csv")

## Combine r/Feminism and the supplementary threads
Run this after both are coded. Same framework and prompt for both, together they are the feminism corpus.
`batch_source` says where a row is from (`original` = r/Feminism, `batch2` = supplementary threads).

In [ ]:
# coded files of r/feminism and the supplementary threads, and the combined output file
FEMINISM_CODED      = Path("path/to/Feminism/coded_final.csv")
SUPPLEMENTARY_CODED = Path("path/to/FeminismBatch2/coded_final.csv")
COMBINED_FILE       = Path("path/to/Feminism_Master_coded_final.csv")

# stack both into one file
master = pd.concat([pd.read_csv(FEMINISM_CODED, low_memory=False).assign(batch_source="original"),
                    pd.read_csv(SUPPLEMENTARY_CODED, low_memory=False).assign(batch_source="batch2")],
                   ignore_index=True)
COMBINED_FILE.parent.mkdir(parents=True, exist_ok=True)
master.to_csv(COMBINED_FILE, index=False)
print(f"feminism corpus: {len(master):,} datapoints")